# 🧹 Pipeline 1: Data Ingestion & Preprocessing Pipeline
**Project:** Build For Bharat 2.0 — CareerGPS Talent Intelligence  
**Track:** SAS & Data Science Track  

### Overview & Objectives
This notebook executes the complete end-to-end data cleaning, sanitization, validation, and feature engineering across all 4 benchmark datasets:
1. **DataScience Jobs** (1,602 employer listings) — Salary parsing, role seniority tagging, salary logic validation.
2. **Analytics Jobs** (15,841 job listings) — Experience parsing, compensation bracket mapping, location standardization, skill extraction.
3. **Junior Data Scientist (JDS) Skill Traits** (139 profiles) — Technical skill ratings (1–5 scale), composite scores, deduplication, target validation.
4. **Senior Data Scientist (SDS) Personality Traits** (161 profiles) — Big Five (OCEAN) psychometric traits, positive trait indices, deduplication.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

raw_dir = os.path.abspath(os.path.join("..", "data", "raw"))
cleaned_dir = os.path.abspath(os.path.join("..", "data", "cleaned"))
os.makedirs(cleaned_dir, exist_ok=True)

print(f"Raw data path: {raw_dir}")
print(f"Cleaned data path: {cleaned_dir}")


Raw data path: C:\Users\AKSHIT\Desktop\Projects\Build For Bharat 2.0\DS\data\raw
Cleaned data path: C:\Users\AKSHIT\Desktop\Projects\Build For Bharat 2.0\DS\data\cleaned


## 1. Preprocessing DataScience Jobs
- Standardize column names.
- Convert raw salary text (e.g. `'4.5L'`) into numeric INR Lakhs (`min_salary_lakhs`, `avg_salary_lakhs`, `max_salary_lakhs`).
- Validate salary monotonicity: ensures $\text{min} \le \text{avg} \le \text{max}$.
- Engineer `is_senior_role` flag based on title keywords (`senior`, `lead`, `architect`).


In [2]:
raw_ds = pd.read_csv(os.path.join(raw_dir, "DataScience Jobs.csv"))
print(f"Raw DataScience Jobs shape: {raw_ds.shape}")
print("Raw columns:", list(raw_ds.columns))

df_ds = raw_ds.copy()
df_ds.columns = [c.strip() for c in df_ds.columns]

for col in ['company_name', 'job_title']:
    df_ds[col] = df_ds[col].astype(str).str.strip()

for col in ['min_salary', 'avg_salary', 'max_salary']:
    clean_col = f"{col}_lakhs"
    df_ds[clean_col] = df_ds[col].astype(str).str.replace('L', '', regex=False).str.strip().astype(float)

df_ds['min_experience_years'] = df_ds['min_experience'].astype(int)
df_ds['num_of_jobs_posted'] = df_ds['num_of_jobs'].astype(int)
df_ds['is_senior_role'] = df_ds['job_title'].str.lower().str.contains('senior|lead|architect', regex=True).astype(int)

# Integrity Check: min <= avg <= max
salary_issues = ((df_ds['min_salary_lakhs'] > df_ds['avg_salary_lakhs']) | (df_ds['avg_salary_lakhs'] > df_ds['max_salary_lakhs'])).sum()
print(f"Salary logic violations (min > avg or avg > max): {salary_issues}")

out_ds = os.path.join(cleaned_dir, "clean_datascience_jobs.csv")
df_ds.to_csv(out_ds, index=False)
print(f"Successfully saved clean DataScience Jobs: {out_ds} ({df_ds.shape[0]} rows, {df_ds.shape[1]} cols)")
df_ds[['job_title', 'company_name', 'min_salary_lakhs', 'avg_salary_lakhs', 'max_salary_lakhs', 'is_senior_role']].head()


Raw DataScience Jobs shape: (1602, 8)
Raw columns: ['reference_no', 'company_name', 'job_title', 'min_experience', 'avg_salary', 'min_salary', 'max_salary', 'num_of_jobs']
Salary logic violations (min > avg or avg > max): 0
Successfully saved clean DataScience Jobs: C:\Users\AKSHIT\Desktop\Projects\Build For Bharat 2.0\DS\data\cleaned\clean_datascience_jobs.csv (1602 rows, 14 cols)


,job_title,company_name,min_salary_lakhs,avg_salary_lakhs,max_salary_lakhs,is_senior_role
0,Data Scientist,TCS,4.5,7.8,16.0,0
1,Data Scientist,Accenture,5.8,12.8,23.0,0
2,Data Scientist,IBM,5.3,13.4,25.0,0
3,Data Scientist,Cognizant,5.0,9.8,18.0,0
4,Data Scientist,Capgemini,4.8,8.6,14.6,0


## 2. Preprocessing Analytics Jobs (15,841 Postings)
- Clean and standardize `job_type`.
- Parse experience string `'X-Y yrs'` into `min_experience_years`, `max_experience_years`, and `avg_experience_years`.
- Map categorical salary brackets (`'0to3'`, `'3to6'`, `'6to10'`, `'10to15'`, `'15to25'`, `'25to50'`) to min, max, and midpoints in INR Lakhs.
- Extract primary location (cleansing multi-city strings).
- Parse and tokenize comma-delimited `key_skills` into clean lowercase tags, and generate binary indicators for top technical skills (Python, SQL, R, Machine Learning, Power BI, Tableau, etc.).


In [3]:
raw_an = pd.read_csv(os.path.join(raw_dir, "Analytics Jobs.csv"))
print(f"Raw Analytics Jobs shape: {raw_an.shape}")

df_an = raw_an.copy()
df_an.columns = [c.strip() for c in df_an.columns]

# Standardize job type
df_an['job_type_clean'] = df_an['job_type'].fillna('Not Specified').astype(str).str.strip()
df_an['job_type_standardized'] = df_an['job_type_clean'].replace({
    'analytics': 'Analytics',
    'ANALYTICS': 'Analytics',
    'analytic': 'Analytics',
    'Analytic': 'Analytics'
})

# Experience range parsing
exp_split = df_an['experience'].astype(str).str.replace(' yrs', '', regex=False).str.split('-', expand=True)
df_an['min_experience_years'] = exp_split[0].str.strip().astype(int)
df_an['max_experience_years'] = exp_split[1].str.strip().astype(int)
df_an['avg_experience_years'] = (df_an['min_experience_years'] + df_an['max_experience_years']) / 2.0

# Salary bracket mapping
salary_bracket_map = {
    '0to3': (0.0, 3.0, 1.5),
    '3to6': (3.0, 6.0, 4.5),
    '6to10': (6.0, 10.0, 8.0),
    '10to15': (10.0, 15.0, 12.5),
    '15to25': (15.0, 25.0, 20.0),
    '25to50': (25.0, 50.0, 37.5)
}
df_an['salary_min_lakhs'] = df_an['salary'].map(lambda x: salary_bracket_map.get(str(x).strip(), (np.nan, np.nan, np.nan))[0])
df_an['salary_max_lakhs'] = df_an['salary'].map(lambda x: salary_bracket_map.get(str(x).strip(), (np.nan, np.nan, np.nan))[1])
df_an['salary_midpoint_lakhs'] = df_an['salary'].map(lambda x: salary_bracket_map.get(str(x).strip(), (np.nan, np.nan, np.nan))[2])

# Location & Text cleaning
df_an['job_desig_clean'] = df_an['job_desig'].fillna('').astype(str).str.strip()
df_an['location_clean'] = df_an['location'].fillna('').astype(str).str.strip()
df_an['primary_location'] = df_an['location_clean'].apply(lambda x: x.split(',')[0].strip() if x else 'Not Specified')

def clean_skill_string(s):
    if pd.isna(s):
        return ""
    s = str(s).replace('...', '').strip()
    tokens = [t.strip().lower() for t in s.split(',') if t.strip() and t.strip() != '...']
    return ", ".join(tokens)

df_an['key_skills_clean'] = df_an['key_skills'].apply(clean_skill_string)
df_an['skill_count'] = df_an['key_skills_clean'].apply(lambda x: len(x.split(',')) if x else 0)

# One-hot/binary indicators for core technical keywords
tech_keywords = ['python', 'sql', 'r', 'sas', 'machine learning', 'analytics', 'data analysis', 'excel', 'tableau', 'power bi', 'hadoop', 'spark', 'aws', 'deep learning']
for kw in tech_keywords:
    col_name = f"has_skill_{kw.replace(' ', '_')}"
    df_an[col_name] = df_an['key_skills_clean'].str.contains(r'\b' + kw + r'\b', regex=True).astype(int)

out_an = os.path.join(cleaned_dir, "clean_analytics_jobs.csv")
df_an.to_csv(out_an, index=False)
print(f"Successfully saved clean Analytics Jobs: {out_an} ({df_an.shape[0]} rows, {df_an.shape[1]} cols)")
df_an[['job_desig_clean', 'primary_location', 'avg_experience_years', 'salary_midpoint_lakhs', 'skill_count']].head()


Raw Analytics Jobs shape: (15841, 8)


Successfully saved clean Analytics Jobs: C:\Users\AKSHIT\Desktop\Projects\Build For Bharat 2.0\DS\data\cleaned\clean_analytics_jobs.csv (15841 rows, 35 cols)


,job_desig_clean,primary_location,avg_experience_years,salary_midpoint_lakhs,skill_count
0,Oracle EBS Technical Consultant,Bengaluru,8.0,8.0,6
1,Staff Software Engineer - Object Oriented Anal...,Bengaluru,10.0,12.5,7
2,Servicenow Developer,Hyderabad,5.5,1.5,3
3,"Software Development Engineering, Analyst",Pune,4.5,8.0,6
4,NVH Engineer - Nastran/ Ls-dyna,Mumbai,7.5,12.5,8


## 3. Preprocessing Junior Data Scientist (JDS) Skill Traits
- Read `JDS Skill Traits.xlsx` (Sheet: `JDS`).
- Audit unique IDs and flag duplicate instances.
- Engineer composite skill metrics:
  - `composite_skill_score`: Mean across all 5 measured competencies.
  - `technical_core_score`: Mean across Coding, Maths-Stats, and AI/ML.


In [4]:
raw_jds = pd.read_excel(os.path.join(raw_dir, "JDS Skill Traits.xlsx"), sheet_name="JDS")
print(f"Raw JDS shape: {raw_jds.shape}")
df_jds = raw_jds.copy()
df_jds.columns = [c.strip() for c in df_jds.columns]

df_jds['is_duplicate_id'] = df_jds.duplicated(subset=['id'], keep=False).astype(int)
print(f"Duplicate IDs identified: {df_jds['is_duplicate_id'].sum()}")

skill_cols = ['big_data_skills', 'maths-stats_skills', 'coding_skills', 'ai_and_ml_skills', 'dashboard_and_storytelling_skills']
df_jds['composite_skill_score'] = df_jds[skill_cols].mean(axis=1).round(3)
df_jds['technical_core_score'] = df_jds[['coding_skills', 'maths-stats_skills', 'ai_and_ml_skills']].mean(axis=1).round(3)

out_jds = os.path.join(cleaned_dir, "clean_jds_skill_traits.csv")
df_jds.to_csv(out_jds, index=False)
print(f"Successfully saved clean JDS: {out_jds} ({df_jds.shape[0]} rows, {df_jds.shape[1]} cols)")
df_jds.head()


Raw JDS shape: (139, 7)
Duplicate IDs identified: 4
Successfully saved clean JDS: C:\Users\AKSHIT\Desktop\Projects\Build For Bharat 2.0\DS\data\cleaned\clean_jds_skill_traits.csv (139 rows, 10 cols)


,id,big_data_skills,maths-stats_skills,coding_skills,ai_and_ml_skills,dashboard_and_storytelling_skills,salary_hike_high_or_low,is_duplicate_id,composite_skill_score,technical_core_score
0,2809,3.6,4.0,4.5,4.8,5.0,1,0,4.38,4.433
1,2231,5.0,3.8,3.3,5.0,4.4,0,0,4.30,4.033
2,3522,4.1,3.8,4.4,4.2,5.0,1,0,4.30,4.133
3,3291,4.4,3.0,3.3,4.6,2.3,0,1,3.52,3.633
4,3424,2.3,2.3,2.6,3.3,3.0,0,0,2.70,2.733


## 4. Preprocessing Senior Data Scientist (SDS) Personality Traits
- Read `SDS Personality Traits.xlsx` (Sheet: `SDS`).
- Standardize Big Five (OCEAN) column names.
- Flag duplicates on employee ID.
- Engineer composite psychometric index: `positive_trait_index` = Conscientiousness + Openness + Extraversion + Agreeableness - Neuroticism.


In [5]:
raw_sds = pd.read_excel(os.path.join(raw_dir, "SDS Personality Traits.xlsx"), sheet_name="SDS")
print(f"Raw SDS shape: {raw_sds.shape}")
df_sds = raw_sds.copy()
df_sds.columns = [c.strip() for c in df_sds.columns]

rename_dict = {
    'extraversion': 'extraversion',
    'openness_to_experience': 'openness_to_experience',
    'agreeableness': 'agreeableness',
    'conscientiousness': 'conscientiousness',
    'neuroticism': 'neuroticism',
    'success_ classification_ high_low': 'success_classification_high_low'
}
df_sds = df_sds.rename(columns=rename_dict)
df_sds['is_duplicate_id'] = df_sds.duplicated(subset=['id'], keep=False).astype(int)
print(f"Duplicate IDs identified: {df_sds['is_duplicate_id'].sum()}")

df_sds['positive_trait_index'] = (
    df_sds['conscientiousness'] + df_sds['openness_to_experience'] + 
    df_sds['extraversion'] + df_sds['agreeableness'] - df_sds['neuroticism']
).round(2)

out_sds = os.path.join(cleaned_dir, "clean_sds_personality_traits.csv")
df_sds.to_csv(out_sds, index=False)
print(f"Successfully saved clean SDS: {out_sds} ({df_sds.shape[0]} rows, {df_sds.shape[1]} cols)")
df_sds.head()


Raw SDS shape: (161, 7)
Duplicate IDs identified: 18
Successfully saved clean SDS: C:\Users\AKSHIT\Desktop\Projects\Build For Bharat 2.0\DS\data\cleaned\clean_sds_personality_traits.csv (161 rows, 9 cols)


,id,neuroticism,extraversion,openness_to_experience,agreeableness,conscientiousness,success_classification_high_low,is_duplicate_id,positive_trait_index
0,8120,33,34,39,45,51,1,0,136
1,8951,50,36,31,59,39,0,0,115
2,8656,38,35,42,39,49,1,1,127
3,8796,41,26,30,40,19,0,0,74
4,8470,20,17,21,31,26,0,0,75


## 5. Preprocessing Summary & Quality Verification
Summary table confirming row counts, columns, and data hygiene across all clean datasets ready for downstream EDA and Modeling.


In [6]:
summary = pd.DataFrame([
    {"Dataset": "DataScience Jobs", "Rows": len(df_ds), "Columns": df_ds.shape[1], "Missing Values": df_ds.isna().sum().sum()},
    {"Dataset": "Analytics Jobs", "Rows": len(df_an), "Columns": df_an.shape[1], "Missing Values": df_an.isna().sum().sum()},
    {"Dataset": "JDS Skill Traits", "Rows": len(df_jds), "Columns": df_jds.shape[1], "Missing Values": df_jds.isna().sum().sum()},
    {"Dataset": "SDS Personality Traits", "Rows": len(df_sds), "Columns": df_sds.shape[1], "Missing Values": df_sds.isna().sum().sum()},
])
print("=== PREPROCESSING PIPELINE SUMMARY ===")
display(summary)


=== PREPROCESSING PIPELINE SUMMARY ===


,Dataset,Rows,Columns,Missing Values
0,DataScience Jobs,1602,14,0
1,Analytics Jobs,15841,35,15520
2,JDS Skill Traits,139,10,0
3,SDS Personality Traits,161,9,0
